#### Packages

In [ ]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import linregress

pd.set_option('display.max_columns', None)
np.random.seed(1)

#### Data

In [ ]:
# Train Data ===================================================
train1 = pd.read_csv('../data/cleaned_data/train_FD001')
train2 = pd.read_csv('../data/cleaned_data/train_FD002')
train3 = pd.read_csv('../data/cleaned_data/train_FD003')
train4 = pd.read_csv('../data/cleaned_data/train_FD004')

train1 = train1.set_index('Engine')
train2 = train2.set_index('Engine')
train3 = train3.set_index('Engine')
train4 = train4.set_index('Engine')

# Test Data =====================================================
test1 = pd.read_csv('../data/cleaned_data/test_FD001')
test2 = pd.read_csv('../data/cleaned_data/test_FD002')
test3 = pd.read_csv('../data/cleaned_data/test_FD003')
test4 = pd.read_csv('../data/cleaned_data/test_FD004')

test1 = test1.set_index('Engine')
test2 = test2.set_index('Engine')
test3 = test3.set_index('Engine')
test4 = test4.set_index('Engine')

#### Feature Engineering

&emsp; During exploratory data analysis (EDA), it was discovered that in datasets possessing 2 fault modes (FD003 and FD004), PC2 tracked a relationship that identified two distinct clusters of faults. I hypothesize that PC2 tracks this latent feature that is not present in the dataset, and including it during training may help the model learn the pattern of the two fault modes (See Figure 1). To avoid data leakage, this PCA feature will be expanding, applied only up to the latest row, with rows beyond any given row not included in the calculation of its PCA features.

<img src="images/figure1.png" title="Figure 1">
<center>Figure 1</center> 

&emsp; Another discovery was severe noise in datasets with six operating conditions. For these datasets, sensor correlation mimicked that of single operating conditions when partitioned (See Figure 2), and so did degradation plotting (See Figure 3). 

<img src="images/figure2.png" width="45%" title="Figure 2"/> <img src="images/figure3.png" width="45%" title="Figure 3"/>
<div style="display: flex; justify-content: flex-start; gap: 10%;">
    <div style="text-align: center; width: 40%;">Figure 2</div>
    <div style="text-align: center; width: 40%;">Figure 3</div>
</div> <br>

Prior to partitioning, these plots were noisy and difficult to read (See Figures 4 and 5). NASA asserts that noise was synthetically added to the operational settings provided in the data, in order to avoid information loss of said noise while including explicit information disclosing the operating condition of the engine. I will also provide a class column during feature engineering.

<img src="images/figure4.png" width="45%" title="Figure 4"/> <img src="images/figure5.png" width="45%" title="Figure 5"/>
<div style="display: flex; justify-content: flex-start; gap: 10%;">
    <div style="text-align: center; width: 40%;">Figure 4</div>
    <div style="text-align: center; width: 40%;">Figure 5</div>
</div> <br>

&emsp; The standard temporal aggregates and lag features, such as expanding mean, standard deviation, delta, and last value, will also be included for each sensor and operational setting during feature engineering.

#### Condition Classes

Classes number 1 - 6 for operating conditions for the training and test sets FD002 and FD004

In [ ]:
def add_conds(df):
    """ Creates and adds a class column for operating conditions 1 - 6

        Args:
            df (pandas.DataFrame): The target DataFrame

        Returns:
            df (pandas.DataFrame): Return DataFrame with op_condition classes.
    """
    df['op_condition'] = df.groupby(
        [df['Setting1'].round(), df['Setting2'].round(), df['Setting3'].round()],
        sort=True
    ).ngroup() + 1

    # move 'class' to second-to-last position
    cols = df.columns.tolist()
    cols.remove('op_condition')
    cols.insert(len(cols) - 1, 'op_condition')
    df = df[cols]

    return df

In [ ]:
# Train Data
train2 = add_conds(train2)
train4 = add_conds(train4)

# Test Data
test2 = add_conds(test2)
test4 = add_conds(test4)

#### Regime Normalization
Rescales sensor features to a common range across operating conditions applicable to datasets FD002 and FD004. 

In [ ]:
def fit_regime_norm_stats(df, sensor_cols, regime_col='op_condition'):
    """Compute per-regime mean/std from TRAIN ONLY. Store and reuse
    on val/test to avoid leakage."""
    stats = {}
    for col in sensor_cols:
        stats[col] = df.groupby(regime_col)[col].agg(['mean', 'std'])
    return stats

def apply_regime_norm(df, stats, sensor_cols, regime_col='op_condition'):
    """Apply previously-fit train regime stats to any dataset (train, val, or test)."""
    df_norm = df.copy()
    for col in sensor_cols:
        regime_stats = stats[col]
        mean_map = df[regime_col].map(regime_stats['mean'])
        std_map = df[regime_col].map(regime_stats['std']).replace(0, 1)
        df_norm[col] = (df[col] - mean_map) / std_map
    return df_norm

In [ ]:
# Train data =============================================
tr2_sensors = train2.iloc[:, 4:25]
tr4_sensors = train4.iloc[:, 4:25]
fd002_norm_stats = fit_regime_norm_stats(train2, tr2_sensors)
fd004_norm_stats = fit_regime_norm_stats(train4, tr4_sensors)
train2 = apply_regime_norm(train2, fd002_norm_stats, tr2_sensors)
train4 = apply_regime_norm(train4, fd004_norm_stats, tr4_sensors)

# Test data =============================================
te2_sensors = test2.iloc[:, 4:25]
te4_sensors = test4.iloc[:, 4:25]
test2 = apply_regime_norm(test2, fd002_norm_stats, te2_sensors)
test4 = apply_regime_norm(test4, fd004_norm_stats, te4_sensors)

#### Principal Component 2

Expanding features for sensor principal component analysis (component 2). Limited to the training and test sets FD003 and FD004.

In [ ]:
def fit_pca(train_df, last_n=60, min_std=1e-3):
    """Fits one frozen scaler and 2-component PCA on training data.

    Standardizing gives each sensor unit variance, so none dominates
    by scale. PCA then finds orthogonal directions of maximum variance
    (eigenvectors of the covariance matrix): PC1 is the largest shared
    drift, PC2 the next largest direction perpendicular to it. Using
    late-life rows makes degradation, not noise, the main variance.
    Fitting once fixes the axes, so a PC2 value means the same thing
    for every engine and cycle.

    Args:
        train_df: Training frame. Engine id is the index, cycle is
            column 0, sensors are columns 4:25.
        last_n: Use each engine's final last_n cycles. None uses all.
        min_std: Drop near-constant sensors, since standardizing would
            inflate their noise.

    Returns:
        Tuple of (cols, scaler, pca).
    """
    cols = [c for c in train_df.columns[4:25] if train_df[c].std() > min_std]
    fit_df = train_df
    if last_n is not None:
        cyc = train_df.iloc[:, 0]
        last = cyc.groupby(level=0).transform("max")
        fit_df = train_df[cyc >= last - last_n]
    scaler = StandardScaler().fit(fit_df[cols])
    pca = PCA(n_components=2).fit(scaler.transform(fit_df[cols]))
    return cols, scaler, pca


def add_pc2(df, cols, scaler, pca):
    """Adds PC2 and its per-engine expanding mean as features.

    Projection is a dot product with a fixed unit vector, so it is
    cheap and identical for train and test. Fault mode is an
    engine-level trait, so a running mean of PC2 estimates it better
    than any single noisy row: noise shrinks roughly as 1/sqrt(n).
    The mean uses only current and past cycles, so it is causal.

    Args:
        df: Frame to transform (train or test), same layout as in
            fit_pca.
        cols: Sensor columns returned by fit_pca.
        scaler: Fitted StandardScaler from fit_pca.
        pca: Fitted PCA from fit_pca.

    Returns:
        out: Copy of df with 'pc2'inserted before the
        last two columns.
    """
    if df.equals(train3) or df.equals(test3):
        place = 1
    else:
        place = 2

    pc2 = pca.transform(scaler.transform(df[cols]))[:, 1]
    out = df.copy()
    out.insert(len(out.columns) - place, "pc2", pc2)
    return out

In [ ]:
cols, scaler, pca = fit_pca(train3)
cols1, scaler1, pca1 = fit_pca(train4)

# Train Data
train3 = add_pc2(train3, cols, scaler, pca)
train4 = add_pc2(train4, cols1, scaler1, pca1)

# Test Data
test3 = add_pc2(test3, cols, scaler, pca)
test4 = add_pc2(test4, cols1, scaler1, pca1)

#### Lag Features

Features that represent a sensor's value before the current value for each column.

In [ ]:
def add_lag(df):
    """ Generates lag features from previous settings and sensors and
        concatenates them into the target DataFrame.

        Args:
            df (pandas.DataFrame): Target DataFrame.

        Returns:
            df (pandas.DataFrame): Return DataFrame with engineered features.
    """
    # Adds a first row with NaN due to no previous readouts.
    new_row = df.groupby(level=0).head(1)
    new_row[:] = np.nan

    # Removes final row because all previous readouts have been used.
    last_row_del = df[df.groupby(level=0).cumcount(ascending=False) != 0]
    lag_df = pd.concat([new_row, last_row_del]).sort_index(kind='stable')

    exclude = [
        "Time",
        "op_condition",
        "RUL",
    ] + [c for c in df.columns if c.endswith("_last")]
    feat_cols = [c for c in df.columns if c not in exclude]

    lag_df = lag_df[feat_cols]
    lag_df.columns = [f"{col}_last" for col in feat_cols]

    # [target df before RUL labels, lag features, target df RUL labels]
    # Keeps RUL labels at the end of the df
    df = pd.concat([df.iloc[:, :-1], lag_df, df.iloc[:, -1]], axis=1)

    return df

In [ ]:
# Train Data
train1 = add_lag(train1)
train2 = add_lag(train2)
train3 = add_lag(train3)
train4 = add_lag(train4)

# Test Data
test1 = add_lag(test1)
test2 = add_lag(test2)
test3 = add_lag(test3)
test4 = add_lag(test4)

#### Temporal Aggregates

Features sensor mean, sensor_std, sensor_min, sensor_max and sensor_delta 

In [ ]:
def slope(x):
    """Vectorized expanding-window OLS slope via Pandas' own rolling corr trick."""
    idx = pd.Series(np.arange(len(x)), index=x.index)
    cov = x.expanding(min_periods=2).cov(idx)
    var = idx.expanding(min_periods=2).var()
    return cov / var


def add_expanding_features(df):
    exclude = [
        "Time",
        "op_condition",
        "RUL",
    ] + [c for c in df.columns if c.endswith("_last")]
    feat_cols = [c for c in df.columns if c not in exclude]

    feats = {}
    for col in feat_cols:
        g = df.groupby(level=0)[col]
        feats[f"{col}_mean"] = g.transform(lambda x: x.expanding().mean())
        feats[f"{col}_slope"] = g.transform(lambda x: slope(x))
        feats[f"{col}_std"] = g.transform(lambda x: x.expanding().std())
        feats[f"{col}_min"] = g.transform(lambda x: x.expanding().min())
        feats[f"{col}_max"] = g.transform(lambda x: x.expanding().max())
        feats[f"{col}_delta"] = feats[f"{col}_max"] - feats[f"{col}_min"]

    feat_df = pd.DataFrame(feats, index=df.index)
    return pd.concat([df.iloc[:, :-2], feat_df, df.iloc[:, -2:]], axis=1)

In [ ]:
# Train Data
train1 = add_expanding_features(train1)
train2 = add_expanding_features(train2)
train3 = add_expanding_features(train3)
train4 = add_expanding_features(train4)

# Test Data
test1 = add_expanding_features(test1)
test2 = add_expanding_features(test2)
test3 = add_expanding_features(test3)
test4 = add_expanding_features(test4)

#### Save Feature Engineered Data

In [ ]:
df_dict = {
    # Training Data
    "../data/FE_data/train_FD001": train1,
    "../data/FE_data/train_FD002": train2,
    "../data/FE_data/train_FD003": train3,
    "../data/FE_data/train_FD004": train4,
    # Test Data
    "../data/FE_data/test_FD001": test1,
    "../data/FE_data/test_FD002": test2,
    "../data/FE_data/test_FD003": test3,
    "../data/FE_data/test_FD004": test4,
}

for url, df in df_dict.items():
    df.reset_index().to_parquet(url, index=False)